# From a glycerol–water mixture to $Re$–$Ca$ space

This notebook reverses the original question. Rather than choosing $(Re,Ca,\theta)$ and searching for a possible fluid, we prescribe:

1. glycerol mass fraction,
2. inclination angle $\theta$,
3. mean film thickness $h_0$,

and calculate the Nusselt mean velocity and resulting Reynolds and capillary numbers.

All mixture properties below are for approximately **20 °C**. Concentration is **glycerol mass percent (wt%)**, not volume percent. Temperature must be stated because the viscosity—especially near pure glycerol—is strongly temperature dependent.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from pathlib import Path

gravity = 9.81
temperature_C = 20.0

# User controls
angles_deg = np.array([6.4, 30.0, 45.0, 60.0, 90.0])
h0_values = np.logspace(np.log10(0.10e-3), np.log10(2.00e-3), 250)
glycerol_wt_values = np.linspace(0.0, 100.0, 201)

# Optional experimental screen (does not alter Re or Ca calculations)
velocity_limits_m_s = (0.0, 0.50)

# Optional DNS parameter-space overlay
dns_Re = np.array([5, 10, 20, 40, 60, 80, 100, 120, 160, 200], dtype=float)
dns_Ca = np.array([0.001, 0.002, 0.005, 0.01, 0.02, 0.04, 0.06, 0.1], dtype=float)


## Governing relations

For a steady laminar Nusselt film down an incline,

$$U_N=\frac{\rho g\sin\theta\,h_0^2}{3\mu}.$$

Using the same definitions as the original notebook,

$$Re=\frac{\rho U_Nh_0}{\mu}
=\frac{\rho^2g\sin\theta\,h_0^3}{3\mu^2},$$

$$Ca=\frac{\mu U_N}{\sigma}
=\frac{\rho g\sin\theta\,h_0^2}{3\sigma}.$$

Thus $Re\propto h_0^3/\mu^2$, whereas $Ca\propto h_0^2/\sigma$. Under this Nusselt scaling, viscosity cancels from $Ca$; glycerol concentration therefore moves a case much more strongly in $Re$ than in $Ca$. These relations assume a fully developed, laminar, gravity-driven film with a shear-free liquid–gas interface and no imposed gas shear or streamwise pressure gradient.

## Glycerol–water properties at 20 °C

The anchor table covers pure water through nominally pure glycerol in 10 wt% increments. Density and viscosity values are representative literature values at 20 °C; surface tension varies comparatively weakly and is interpolated between representative aqueous-solution values. Shape-preserving interpolation avoids overshoot between anchors.

For publication-quality property work, replace the anchors with values from the exact glycerol grade and measured laboratory temperature. Useful sources are Cheng (2008) for the full-range viscosity correlation and Takamura, Fischer & Morrow (2012) for aqueous-glycerol properties.

In [ ]:
property_table = pd.DataFrame({
    "glycerol_wt_percent": [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    "rho_kg_m3":           [998.2, 1023.3, 1047.6, 1072.7, 1099.6, 1128.7, 1158.6, 1187.9, 1217.0, 1244.6, 1261.3],
    "mu_mPa_s":            [1.002, 1.31, 1.76, 2.50, 3.72, 6.00, 10.8, 22.5, 60.1, 219.0, 1412.0],
    "sigma_mN_m":          [72.75, 72.0, 71.3, 70.5, 69.7, 68.8, 67.9, 66.9, 65.9, 64.8, 63.4],
})
property_table


In [ ]:
rho_interp = PchipInterpolator(property_table["glycerol_wt_percent"], property_table["rho_kg_m3"])

# Interpolate log(mu), because viscosity changes by over three orders of magnitude.
log_mu_interp = PchipInterpolator(
    property_table["glycerol_wt_percent"],
    np.log(property_table["mu_mPa_s"] * 1e-3),
)
sigma_interp = PchipInterpolator(
    property_table["glycerol_wt_percent"],
    property_table["sigma_mN_m"] * 1e-3,
)

def glycerol_water_properties(glycerol_wt_percent):
    # Return rho [kg/m3], mu [Pa s], sigma [N/m] at about 20 C.
    w = np.asarray(glycerol_wt_percent, dtype=float)
    if np.any((w < 0.0) | (w > 100.0)):
        raise ValueError("glycerol_wt_percent must be between 0 and 100")
    return rho_interp(w), np.exp(log_mu_interp(w)), sigma_interp(w)

property_table.round({"rho_kg_m3": 1, "mu_mPa_s": 3, "sigma_mN_m": 2})


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
w_plot = glycerol_wt_values
rho_plot, mu_plot, sigma_plot = glycerol_water_properties(w_plot)

axes[0].plot(w_plot, rho_plot)
axes[0].set_ylabel(r"$\rho$ [kg m$^{-3}$]")
axes[1].semilogy(w_plot, 1e3 * mu_plot)
axes[1].set_ylabel(r"$\mu$ [mPa s]")
axes[2].plot(w_plot, 1e3 * sigma_plot)
axes[2].set_ylabel(r"$\sigma$ [mN m$^{-1}$]")
for ax in axes:
    ax.set_xlabel("Glycerol concentration [wt%]")
    ax.grid(alpha=0.25)
plt.show()


## Calculate one physical case

In [ ]:
def evaluate_physical_case(glycerol_wt_percent, angle_deg, h0, gravity=9.81):
    # Calculate properties, Nusselt velocity, Re and Ca for one physical case.
    if h0 <= 0:
        raise ValueError("h0 must be positive")
    if not (0 < angle_deg <= 90):
        raise ValueError("angle_deg must lie in (0, 90]")

    rho, mu, sigma = glycerol_water_properties(glycerol_wt_percent)
    sin_theta = np.sin(np.radians(angle_deg))
    velocity = rho * gravity * sin_theta * h0**2 / (3.0 * mu)
    Re = rho * velocity * h0 / mu
    Ca = mu * velocity / sigma

    return {
        "glycerol_wt_percent": float(glycerol_wt_percent),
        "angle_deg": float(angle_deg),
        "h0_m": float(h0),
        "rho_kg_m3": float(rho),
        "mu_Pa_s": float(mu),
        "sigma_N_m": float(sigma),
        "velocity_m_s": float(velocity),
        "Re": float(Re),
        "Ca": float(Ca),
    }

chosen_case = evaluate_physical_case(
    glycerol_wt_percent=60.0,
    angle_deg=6.4,
    h0=1.0e-3,
)
pd.Series(chosen_case)


## Sweep concentration, thickness and angle

In [ ]:
def build_parameter_map(glycerol_wt_values, h0_values, angles_deg, gravity=9.81):
    rows = []
    for angle_deg in np.atleast_1d(angles_deg):
        for glycerol_wt in np.atleast_1d(glycerol_wt_values):
            rho, mu, sigma = glycerol_water_properties(glycerol_wt)
            sin_theta = np.sin(np.radians(angle_deg))
            U = rho * gravity * sin_theta * h0_values**2 / (3.0 * mu)
            Re = rho * U * h0_values / mu
            Ca = mu * U / sigma
            for h0, u, re, ca in zip(h0_values, U, Re, Ca):
                rows.append({
                    "glycerol_wt_percent": glycerol_wt,
                    "angle_deg": angle_deg,
                    "h0_m": h0,
                    "h0_mm": 1e3 * h0,
                    "rho_kg_m3": rho,
                    "mu_Pa_s": mu,
                    "sigma_N_m": sigma,
                    "velocity_m_s": u,
                    "Re": re,
                    "Ca": ca,
                    "velocity_realistic": velocity_limits_m_s[0] <= u <= velocity_limits_m_s[1],
                })
    return pd.DataFrame(rows)

parameter_map = build_parameter_map(glycerol_wt_values, h0_values, angles_deg)
parameter_map.head()


In [ ]:
summary_by_angle = (
    parameter_map.groupby("angle_deg")
    .agg(Re_min=("Re", "min"), Re_max=("Re", "max"),
         Ca_min=("Ca", "min"), Ca_max=("Ca", "max"),
         U_min_m_s=("velocity_m_s", "min"), U_max_m_s=("velocity_m_s", "max"))
)
summary_by_angle


In [ ]:
# Ranges after applying only the selected experimental velocity screen
screened_map = parameter_map[parameter_map["velocity_realistic"]]
screened_summary_by_angle = (
    screened_map.groupby("angle_deg")
    .agg(Re_min=("Re", "min"), Re_max=("Re", "max"),
         Ca_min=("Ca", "min"), Ca_max=("Ca", "max"),
         U_min_m_s=("velocity_m_s", "min"), U_max_m_s=("velocity_m_s", "max"))
)
screened_summary_by_angle


### Heat maps in physical-input space

These plots answer: *for a chosen angle, which $Re$ and $Ca$ result from each glycerol concentration and film thickness?*

In [ ]:
chosen_angle = 6.4
angle_map = parameter_map[parameter_map["angle_deg"] == chosen_angle]

def pivot_value(column):
    return angle_map.pivot(index="h0_mm", columns="glycerol_wt_percent", values=column)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
for ax, column, title, log_color in zip(
    axes,
    ["Re", "Ca", "velocity_m_s"],
    [r"$Re$", r"$Ca$", r"$U_N$ [m/s]"],
    [True, True, True],
):
    matrix = pivot_value(column)
    values = np.log10(matrix.values) if log_color else matrix.values
    mesh = ax.pcolormesh(matrix.columns, matrix.index, values, shading="auto")
    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label(rf"$\log_{{10}}$({title})" if log_color else title)
    ax.set_xlabel("Glycerol concentration [wt%]")
    ax.set_ylabel(r"Film thickness $h_0$ [mm]")
    ax.set_title(rf"{title}, $\theta={chosen_angle:g}^\circ$")
plt.show()


### Curves in the $Re$–$Ca$ plane

Each curve holds concentration fixed and varies thickness from 0.10 to 2.00 mm. Markers indicate selected thicknesses. The black crosses show the discrete DNS design points.

In [ ]:
chosen_concentrations = [0, 20, 40, 60, 80, 90, 100]
marker_h0_mm = np.array([0.10, 0.25, 0.50, 1.00, 1.50, 2.00])

fig, axes = plt.subplots(1, len(angles_deg), figsize=(20, 4.3), sharex=True, sharey=True, constrained_layout=True)
colors = plt.cm.viridis(np.linspace(0, 1, len(chosen_concentrations)))

for ax, angle_deg in zip(np.atleast_1d(axes), angles_deg):
    for color, wt in zip(colors, chosen_concentrations):
        line = parameter_map[(parameter_map["angle_deg"] == angle_deg) & np.isclose(parameter_map["glycerol_wt_percent"], wt)]
        ax.plot(line["Re"], line["Ca"], color=color, label=f"{wt} wt%")
        for hm in marker_h0_mm:
            point = evaluate_physical_case(wt, angle_deg, hm * 1e-3)
            ax.scatter(point["Re"], point["Ca"], color=color, s=13)

    Re_grid, Ca_grid = np.meshgrid(dns_Re, dns_Ca)
    ax.scatter(Re_grid, Ca_grid, marker="x", color="black", s=20, alpha=0.55, label="DNS grid" if angle_deg == angles_deg[0] else None)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(which="both", alpha=0.2)
    ax.set_title(rf"$\theta={angle_deg:g}^\circ$")
    ax.set_xlabel(r"$Re$")
axes[0].set_ylabel(r"$Ca$")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles[:len(chosen_concentrations)] + [axes[0].collections[-1]],
           labels[:len(chosen_concentrations)] + ["DNS grid"], loc="outside lower center", ncol=8)
plt.show()


## Find physical realizations near a target DNS point

This optional query does not drive the calculation. It filters the already-generated physical sweep to show which concentration/thickness combinations land near a selected DNS $(Re,Ca)$ point.

In [ ]:
def find_nearby_realizations(parameter_map, target_Re, target_Ca, angle_deg,
                             log_distance_tolerance=0.10, max_rows=20):
    subset = parameter_map[parameter_map["angle_deg"] == angle_deg].copy()
    subset["log_distance"] = np.sqrt(
        np.log10(subset["Re"] / target_Re)**2 +
        np.log10(subset["Ca"] / target_Ca)**2
    )
    subset["Re_error_percent"] = 100 * (subset["Re"] / target_Re - 1)
    subset["Ca_error_percent"] = 100 * (subset["Ca"] / target_Ca - 1)
    return subset[subset["log_distance"] <= log_distance_tolerance].nsmallest(max_rows, "log_distance")

nearby = find_nearby_realizations(parameter_map, target_Re=20, target_Ca=0.01, angle_deg=6.4)
nearby[["glycerol_wt_percent", "h0_mm", "velocity_m_s", "Re", "Ca",
        "Re_error_percent", "Ca_error_percent"]]


## Export the sweep

Uncomment the final line to save the full table. The filename records the assumed temperature.

In [ ]:
output_file = Path(f"glycerol_water_Re_Ca_map_{temperature_C:g}C.csv")
# parameter_map.to_csv(output_file, index=False)
output_file


## Interpretation and limitations

- Near pure water, modest sub-millimetric films can produce comparatively large $Re$. Adding glycerol rapidly suppresses $Re$ because of the $\mu^{-2}$ dependence.
- At fixed $h_0$ and $\theta$, changing concentration moves $Ca$ only modestly because density and surface tension vary far less than viscosity.
- “Pure glycerol” is hygroscopic in practice. Record actual water content and temperature; both can materially affect viscosity.
- If gas shear, imposed pressure gradient, finite-amplitude waves, heating, or evaporation is important, use the actual mean flow rate/velocity in $Re$ and $Ca$ rather than the shear-free Nusselt velocity.

References: N.-S. Cheng, “Formula for the Viscosity of a Glycerol–Water Mixture,” *Industrial & Engineering Chemistry Research* 47 (2008), 3285–3288, DOI: 10.1021/ie071349z. K. Takamura, H. Fischer, and N. R. Morrow, “Physical properties of aqueous glycerol solutions,” *Journal of Petroleum Science and Engineering* 98–99 (2012), 50–60.

In [ ]:
# Relative tolerances for declaring a DNS point realistic
Re_tolerance = 0.10
Ca_tolerance = 0.10

dns_match_rows = []

for angle_deg in angles_deg:

    # Only consider cases satisfying the velocity constraint
    candidates = parameter_map[
        (parameter_map["angle_deg"] == angle_deg)
        & parameter_map["velocity_realistic"]
    ].copy()

    for Re_target in dns_Re:
        for Ca_target in dns_Ca:

            # Find the closest physical case in logarithmic Re-Ca space
            log_distance = np.sqrt(
                np.log10(candidates["Re"] / Re_target)**2
                + np.log10(candidates["Ca"] / Ca_target)**2
            )

            best = candidates.loc[log_distance.idxmin()]

            Re_error = abs(
                best["Re"] / Re_target - 1.0
            )

            Ca_error = abs(
                best["Ca"] / Ca_target - 1.0
            )

            realistic = (
                (Re_error <= Re_tolerance)
                and (Ca_error <= Ca_tolerance)
            )

            dns_match_rows.append({
                "angle_deg": angle_deg,
                "Re_target": Re_target,
                "Ca_target": Ca_target,
                "realistic": realistic,

                # Only retain the physical realization
                # when it satisfies both tolerances
                "glycerol_wt_percent": (
                    best["glycerol_wt_percent"]
                    if realistic else np.nan
                ),

                "h0_mm": (
                    best["h0_mm"]
                    if realistic else np.nan
                ),

                "velocity_m_s": (
                    best["velocity_m_s"]
                    if realistic else np.nan
                ),

                "Re_actual": (
                    best["Re"]
                    if realistic else np.nan
                ),

                "Ca_actual": (
                    best["Ca"]
                    if realistic else np.nan
                ),

                "Re_error_percent": 100.0 * Re_error,
                "Ca_error_percent": 100.0 * Ca_error,
            })


dns_match_df = pd.DataFrame(dns_match_rows)

# print(
#     "Realistic DNS points:",
#     dns_match_df["realistic"].sum(),
#     "/",
#     len(dns_match_df),
# )

dns_match_df

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

cmap = plt.cm.viridis

concentration_norm = Normalize(
    vmin=0.0,
    vmax=100.0,
)

fig, axes = plt.subplots(
    1,
    len(angles_deg),
    figsize=(20, 4.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

axes = np.atleast_1d(axes)

for ax, angle_deg in zip(axes, angles_deg):

    angle_df = dns_match_df[
        dns_match_df["angle_deg"] == angle_deg
    ]

    realistic_df = angle_df[
        angle_df["realistic"]
    ]

    unrealistic_df = angle_df[
        ~angle_df["realistic"]
    ]

    # Unrealistic DNS points
    # Unrealistic DNS points: red crosses
    ax.scatter(
        unrealistic_df["Re_target"],
        unrealistic_df["Ca_target"],
        color="red",
        marker="x",
        s=90,
        linewidth=2.0,
        zorder=4,
    )

    # Realistic DNS points, colored by glycerol concentration
    ax.scatter(
        realistic_df["Re_target"],
        realistic_df["Ca_target"],
        c=realistic_df["glycerol_wt_percent"],
        cmap=cmap,
        norm=concentration_norm,
        marker="s",
        s=90,
        edgecolor="black",
        linewidth=0.5,
        zorder=3,
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xticks(dns_Re)
    ax.set_xticklabels(
        [f"{value:g}" for value in dns_Re],
        rotation=45,
    )

    ax.set_yticks(dns_Ca)
    ax.set_yticklabels(
        [f"{value:g}" for value in dns_Ca]
    )

    ax.set_xlabel(r"$Re$")
    ax.set_title(
        rf"$\theta={angle_deg:g}^\circ$"
    )

    ax.grid(
        which="both",
        alpha=0.20,
    )

axes[0].set_ylabel(r"$Ca$")

# Shared concentration colorbar
scalar_mappable = plt.cm.ScalarMappable(
    norm=concentration_norm,
    cmap=cmap,
)

colorbar = fig.colorbar(
    scalar_mappable,
    ax=axes,
    pad=0.02,
)

colorbar.set_label(
    "Matched glycerol concentration [wt%]"
)

# Legend for black points
unrealistic_handle = Line2D(
    [0],
    [0],
    marker="x",
    linestyle="none",
    color="red",
    markeredgewidth=2.0,
    markersize=9,
    label="No realistic match",
)

fig.legend(
    handles=[unrealistic_handle],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
)

plt.show()

In [ ]:
# Physical-property ranges for the realistic DNS realizations

realistic_cases = dns_match_df[
    dns_match_df["realistic"]
].copy()

if realistic_cases.empty:

    print("No DNS points satisfy the selected realism criteria.")

else:

    # Recover mixture properties from matched glycerol concentrations
    rho, mu, sigma = glycerol_water_properties(
        realistic_cases["glycerol_wt_percent"].to_numpy()
    )

    realistic_cases["rho_kg_m3"] = rho
    realistic_cases["mu_mPa_s"] = 1e3 * mu
    realistic_cases["sigma_mN_m"] = 1e3 * sigma

    physical_range_summary = pd.DataFrame({
        "Minimum": [
            realistic_cases["glycerol_wt_percent"].min(),
            realistic_cases["h0_mm"].min(),
            realistic_cases["velocity_m_s"].min(),
            realistic_cases["rho_kg_m3"].min(),
            realistic_cases["mu_mPa_s"].min(),
            realistic_cases["sigma_mN_m"].min(),
            realistic_cases["Re_actual"].min(),
            realistic_cases["Ca_actual"].min(),
        ],

        "Maximum": [
            realistic_cases["glycerol_wt_percent"].max(),
            realistic_cases["h0_mm"].max(),
            realistic_cases["velocity_m_s"].max(),
            realistic_cases["rho_kg_m3"].max(),
            realistic_cases["mu_mPa_s"].max(),
            realistic_cases["sigma_mN_m"].max(),
            realistic_cases["Re_actual"].max(),
            realistic_cases["Ca_actual"].max(),
        ],
    },
    index=[
        "Glycerol concentration [wt%]",
        "Film thickness h0 [mm]",
        "Nusselt velocity U_N [m/s]",
        "Density rho [kg/m³]",
        "Dynamic viscosity mu [mPa s]",
        "Surface tension sigma [mN/m]",
        "Actual Reynolds number",
        "Actual Capillary number",
    ])

    display(
        physical_range_summary.style.format({
            "Minimum": "{:.5g}",
            "Maximum": "{:.5g}",
        })
    )

In [ ]:
# ---------------------------------------------------------
# Choose the target DNS point
# ---------------------------------------------------------

target_Re = 120.0
target_Ca = 0.01
target_angle_deg = 6.4

# Maximum acceptable relative errors
Re_tolerance = 0.10
Ca_tolerance = 0.10

In [ ]:



# ---------------------------------------------------------
# Find the closest realistic physical realization
# ---------------------------------------------------------

candidates = parameter_map[
    (parameter_map["angle_deg"] == target_angle_deg)
    & parameter_map["velocity_realistic"]
].copy()

if candidates.empty:

    print(
        f"No physical cases are available for "
        f"theta = {target_angle_deg:g} degrees."
    )

else:

    # Distance in logarithmic Re-Ca space
    candidates["log_distance"] = np.sqrt(
        np.log10(candidates["Re"] / target_Re)**2
        + np.log10(candidates["Ca"] / target_Ca)**2
    )

    best = candidates.loc[
        candidates["log_distance"].idxmin()
    ].copy()

    Re_error = abs(
        best["Re"] / target_Re - 1.0
    )

    Ca_error = abs(
        best["Ca"] / target_Ca - 1.0
    )

    realistic_match = (
        (Re_error <= Re_tolerance)
        and (Ca_error <= Ca_tolerance)
    )

    if realistic_match:

        match_summary = pd.Series({
            "Target Re": target_Re,
            "Actual Re": best["Re"],
            "Re error [%]": 100.0 * Re_error,

            "Target Ca": target_Ca,
            "Actual Ca": best["Ca"],
            "Ca error [%]": 100.0 * Ca_error,

            "Inclination angle [deg]": target_angle_deg,
            "Glycerol concentration [wt%]":
                best["glycerol_wt_percent"],

            "Film thickness h0 [mm]":
                best["h0_mm"],

            "Nusselt velocity U_N [m/s]":
                best["velocity_m_s"],

            "Density rho [kg/m³]":
                best["rho_kg_m3"],

            "Dynamic viscosity mu [mPa s]":
                1e3 * best["mu_Pa_s"],

            "Surface tension sigma [mN/m]":
                1e3 * best["sigma_N_m"],
        })

        print("A realistic physical match was found.\n")

        display(
            match_summary.to_frame(
                name="Value"
            ).style.format({
                "Value": "{:.6g}"
            })
        )

    else:

        print("No realistic match was found within the selected tolerances.")
        print()
        print("Closest available physical case:")
        print(
            f"  glycerol concentration = "
            f"{best['glycerol_wt_percent']:.2f} wt%"
        )
        print(
            f"  film thickness         = "
            f"{best['h0_mm']:.4f} mm"
        )
        print(
            f"  velocity               = "
            f"{best['velocity_m_s']:.5f} m/s"
        )
        print(
            f"  Re                     = "
            f"{best['Re']:.5g} "
            f"(error = {100 * Re_error:.1f}%)"
        )
        print(
            f"  Ca                     = "
            f"{best['Ca']:.5g} "
            f"(error = {100 * Ca_error:.1f}%)"
        )

Now I want to create folders and run sims based on these realistic values.

In [ ]:
# ---------------------------------------------------------
# Requested DNS parameter space
# ---------------------------------------------------------

selected_angles_deg = np.array(
    [6.4, 45.0, 90.0],
    dtype=float,
)

selected_Re = np.array(
    [10, 20, 40, 80, 120, 200],
    dtype=float,
)

selected_Ca = np.array(
    [
        0.001,
        0.002,
        0.005,
        0.01,
        0.02,
        0.04,
        0.06,
        0.1,
    ],
    dtype=float,
)


# ---------------------------------------------------------
# Physical-realism criteria
# ---------------------------------------------------------

Re_tolerance = 0.10
Ca_tolerance = 0.10

maximum_velocity = 0.50          # m/s
maximum_glycerol_wt = 90.0       # wt%

realistic_case_rows = []


for angle_deg in selected_angles_deg:

    # Physical cases available at this inclination angle
    candidates = parameter_map[
        (parameter_map["angle_deg"] == angle_deg)
        & (
            parameter_map["velocity_m_s"]
            <= maximum_velocity
        )
        & (
            parameter_map["glycerol_wt_percent"]
            <= maximum_glycerol_wt
        )
    ].copy()

    if candidates.empty:
        raise RuntimeError(
            f"No physical candidates are available at "
            f"theta = {angle_deg:g} degrees."
        )

    for Re_target in selected_Re:
        for Ca_target in selected_Ca:

            # Find closest case in logarithmic Re-Ca space
            log_distance = np.sqrt(
                np.log10(
                    candidates["Re"] / Re_target
                )**2
                +
                np.log10(
                    candidates["Ca"] / Ca_target
                )**2
            )

            best = candidates.loc[
                log_distance.idxmin()
            ]

            Re_error = abs(
                best["Re"] / Re_target - 1.0
            )

            Ca_error = abs(
                best["Ca"] / Ca_target - 1.0
            )

            realistic = (
                (Re_error <= Re_tolerance)
                and
                (Ca_error <= Ca_tolerance)
            )

            realistic_case_rows.append({
                "Re_target": Re_target,
                "Ca_target": Ca_target,
                "angle_deg": angle_deg,
                "realistic": realistic,

                "Re_actual": best["Re"],
                "Ca_actual": best["Ca"],

                "Re_error_percent":
                    100.0 * Re_error,

                "Ca_error_percent":
                    100.0 * Ca_error,

                "glycerol_wt_percent":
                    best["glycerol_wt_percent"],

                "h0_mm":
                    best["h0_mm"],

                "velocity_m_s":
                    best["velocity_m_s"],

                "rho_kg_m3":
                    best["rho_kg_m3"],

                "mu_Pa_s":
                    best["mu_Pa_s"],

                "sigma_N_m":
                    best["sigma_N_m"],
            })


selected_case_df = pd.DataFrame(
    realistic_case_rows
)

realistic_case_df = (
    selected_case_df[
        selected_case_df["realistic"]
    ]
    .copy()
    .reset_index(drop=True)
)

unrealistic_case_df = (
    selected_case_df[
        ~selected_case_df["realistic"]
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    f"Total requested cases: "
    f"{len(selected_case_df)}"
)

print(
    f"Realistic cases: "
    f"{len(realistic_case_df)}"
)

print(
    f"Unrealistic cases: "
    f"{len(unrealistic_case_df)}"
)


display(
    realistic_case_df[
        [
            "Re_target",
            "Ca_target",
            "angle_deg",
            "glycerol_wt_percent",
            "h0_mm",
            "velocity_m_s",
            "rho_kg_m3",
            "mu_Pa_s",
            "Re_actual",
            "Ca_actual",
            "Re_error_percent",
            "Ca_error_percent",
        ]
    ]
)

In [ ]:
display(
    unrealistic_case_df[
        [
            "Re_target",
            "Ca_target",
            "angle_deg",
            "Re_actual",
            "Ca_actual",
            "Re_error_percent",
            "Ca_error_percent",
        ]
    ]
)

In [ ]:
from pathlib import Path
import shutil
import re

# ---------------------------------------------------------
# Source and destination
# ---------------------------------------------------------

simulation_root = Path(
    "/blue/bala1s/krishnap.kalivel/TLFHydrodynamics/thinfilm-closure-ml/datasets/BasC/param_space"
)

files_to_copy = [
    simulation_root / "filmflow",
    simulation_root / "filmflow_restart",
    simulation_root / "run.sh",
]

In [ ]:


# ---------------------------------------------------------
# Validate source files before creating anything
# ---------------------------------------------------------

missing_files = [
    file for file in files_to_copy
    if not file.is_file()
]

if missing_files:
    missing_text = "\n".join(
        str(file) for file in missing_files
    )

    raise FileNotFoundError(
        "The following source files were not found:\n"
        + missing_text
    )


# ---------------------------------------------------------
# Folder-name formatting
# ---------------------------------------------------------

def format_Re_for_folder(Re):
    """
    Examples:
        10  -> 010
        80  -> 080
        120 -> 120
        200 -> 200
    """
    return f"{int(round(Re)):03d}"


def format_Ca_for_folder(Ca):
    """
    Examples:
        0.001 -> 0_001
        0.005 -> 0_005
        0.010 -> 0_010
        0.100 -> 0_100
    """
    return f"{Ca:.3f}".replace(".", "_")


def format_angle_for_folder(angle_deg):
    """
    Examples:
        6.4 -> 6_4
        45  -> 45
        90  -> 90
    """
    if float(angle_deg).is_integer():
        return str(int(angle_deg))

    return (
        f"{angle_deg:g}"
        .replace(".", "_")
    )


def make_case_directory_name(
    Re,
    Ca,
    angle_deg,
):
    return (
        f"Re{format_Re_for_folder(Re)}"
        f"_Ca{format_Ca_for_folder(Ca)}"
        f"_theta{format_angle_for_folder(angle_deg)}"
    )


In [ ]:
import re
import shutil
import pandas as pd


# ---------------------------------------------------------
# Validate template files before creating directories
# ---------------------------------------------------------

source_filmflow = simulation_root / "filmflow"
source_filmflow_restart = simulation_root / "filmflow_restart"
source_run_file = simulation_root / "run.sh"
source_makefile = simulation_root / "Makefile"

required_source_files = [
    source_filmflow,
    source_filmflow_restart,
    source_run_file,
]

missing_source_files = [
    path for path in required_source_files
    if not path.is_file()
]

if missing_source_files:
    raise FileNotFoundError(
        "Missing source files:\n"
        + "\n".join(
            str(path)
            for path in missing_source_files
        )
    )


# Read the run script only once
run_template = source_run_file.read_text(
    encoding="utf-8"
)


# ---------------------------------------------------------
# Create realistic simulation cases
# ---------------------------------------------------------

created_case_rows = []
skipped_case_rows = []

for _, case in realistic_case_df.iterrows():

    Re_target = float(case["Re_target"])
    Ca_target = float(case["Ca_target"])
    angle_deg = float(case["angle_deg"])

    directory_name = make_case_directory_name(
        Re=Re_target,
        Ca=Ca_target,
        angle_deg=angle_deg,
    )

    case_directory = (
        simulation_root / directory_name
    )


    # -----------------------------------------------------
    # Skip existing directories completely
    # -----------------------------------------------------

    if case_directory.exists():

        print(
            f"Skipping existing directory: "
            f"{directory_name}"
        )

        skipped_case_rows.append({
            "directory": directory_name,
            "path": str(case_directory),
            "reason": "Directory already exists",
        })

        continue


    # -----------------------------------------------------
    # Prepare the modified run.sh contents
    # -----------------------------------------------------

    job_name = (
        f"R{format_Re_for_folder(Re_target)}"
        f"_C{format_Ca_for_folder(Ca_target)}"
        f"_T{format_angle_for_folder(angle_deg)}"
    )

    modified_run_text, replacements = re.subn(
        pattern=(
            r"^#SBATCH[ \t]+"
            r"--job-name(?:=|[ \t]+)\S+"
        ),
        repl=f"#SBATCH --job-name={job_name}",
        string=run_template,
        count=1,
        flags=re.MULTILINE,
    )

    if replacements != 1:
        raise RuntimeError(
            "Could not find exactly one "
            "'#SBATCH --job-name' directive in "
            f"{source_run_file}"
        )


    # -----------------------------------------------------
    # Matched physical properties
    # -----------------------------------------------------

    rho_l = float(case["rho_kg_m3"])
    mu_l = float(case["mu_Pa_s"])

    glycerol_wt = float(
        case["glycerol_wt_percent"]
    )

    h0_mm = float(case["h0_mm"])
    velocity = float(case["velocity_m_s"])


    # -----------------------------------------------------
    # Prepare params.in contents
    # -----------------------------------------------------

    params_text = f"""LX = 800
MAXLEVEL = 14
AR = 128
Zoom = 128
CFL = 0.1
CA = {Ca_target:.6g}
RE = {Re_target:.1f}
ANGLE_DEG = {angle_deg:.6g}
RHO_L = {rho_l:.8g}
MU_L = {mu_l:.8g}
T_OUT = 10
T_DUMP = 50
T_END = 2000
F0_NOISE = 0.025
FCUT_DIM = 28.0
NOISE_SEED = 12345
MNOISE = 1000
"""


    # -----------------------------------------------------
    # Create the directory
    # -----------------------------------------------------

    case_directory.mkdir(
        parents=False,
        exist_ok=False,
    )


    # -----------------------------------------------------
    # Copy the executables
    # -----------------------------------------------------

    shutil.copy2(
        source_filmflow,
        case_directory / "filmflow",
    )

    shutil.copy2(
        source_filmflow_restart,
        case_directory / "filmflow_restart",
    )
    
    shutil.copy2(
        source_makefile,
        case_directory / "Makefile",
    )


    # -----------------------------------------------------
    # Write the customized run.sh
    # -----------------------------------------------------

    destination_run_file = (
        case_directory / "run.sh"
    )

    destination_run_file.write_text(
        modified_run_text,
        encoding="utf-8",
    )

    # Give the new run.sh the permissions of the template
    shutil.copymode(
        source_run_file,
        destination_run_file,
    )


    # -----------------------------------------------------
    # Write params.in
    # -----------------------------------------------------

    params_file = (
        case_directory / "params.in"
    )

    params_file.write_text(
        params_text,
        encoding="utf-8",
    )


    # -----------------------------------------------------
    # Ensure scripts and executables are executable
    # -----------------------------------------------------

    for executable_name in [
        "filmflow",
        "filmflow_restart",
    ]:

        executable_path = (
            case_directory / executable_name
        )

        executable_path.chmod(
            executable_path.stat().st_mode
            | 0o111
        )


    # -----------------------------------------------------
    # Record the newly created case
    # -----------------------------------------------------

    created_case_rows.append({
        "directory": directory_name,
        "job_name": job_name,
        "path": str(case_directory),
        "Re": Re_target,
        "Ca": Ca_target,
        "angle_deg": angle_deg,
        "glycerol_wt_percent": glycerol_wt,
        "h0_mm": h0_mm,
        "velocity_m_s": velocity,
        "rho_kg_m3": rho_l,
        "mu_Pa_s": mu_l,
    })

    print(
        f"Created: {directory_name} "
        f"[job name: {job_name}]"
    )


# ---------------------------------------------------------
# Summarize the result
# ---------------------------------------------------------

created_case_df = pd.DataFrame(
    created_case_rows
)

skipped_case_df = pd.DataFrame(
    skipped_case_rows
)

print()

print(
    f"New directories created: "
    f"{len(created_case_df)}"
)

print(
    f"Existing directories skipped: "
    f"{len(skipped_case_df)}"
)

if not created_case_df.empty:
    print("\nCreated cases:")
    display(created_case_df)

if not skipped_case_df.empty:
    print("\nSkipped cases:")
    display(skipped_case_df)

In [ ]:
from pathlib import Path
import shutil
import pandas as pd


simulation_root = Path(
    "/blue/bala1s/krishnap.kalivel/"
    "TLFHydrodynamics/thinfilm-closure-ml/"
    "datasets/BasC/param_space"
).resolve()

# Safety switch: keep False for the first run
DELETE_CONFIRMED = False


# ---------------------------------------------------------
# Identify simulation directories with no HDF5 data
# ---------------------------------------------------------

empty_case_rows = []

for case_directory in sorted(
    simulation_root.glob("Re*_Ca*_theta*")
):
    if not case_directory.is_dir():
        continue

    # Do not follow or delete symbolic links
    if case_directory.is_symlink():
        continue

    case_directory = case_directory.resolve()

    # Verify that the candidate is directly inside simulation_root
    if case_directory.parent != simulation_root:
        raise RuntimeError(
            f"Unsafe deletion candidate detected: "
            f"{case_directory}"
        )

    output_directory = (
        case_directory / "output"
    )

    if output_directory.is_dir():
        h5_files = list(
            output_directory.glob("*.h5")
        )
        output_exists = True
    else:
        h5_files = []
        output_exists = False

    if len(h5_files) == 0:
        empty_case_rows.append({
            "directory":
                case_directory.name,

            "path":
                str(case_directory),

            "output_directory_exists":
                output_exists,

            "num_h5_files":
                0,
        })


empty_case_df = pd.DataFrame(
    empty_case_rows
)

print(
    f"Found {len(empty_case_df)} simulation "
    f"directories with no HDF5 data."
)

display(empty_case_df)




In [ ]:
DELETE_CONFIRMED = True

In [ ]:
DELETE_CONFIRMED = False

In [ ]:
# ---------------------------------------------------------
# Delete only after explicit confirmation
# ---------------------------------------------------------

if not DELETE_CONFIRMED:

    print()
    print("PREVIEW ONLY: no directories were deleted.")
    print(
        "Inspect empty_case_df, then set "
        "DELETE_CONFIRMED = True and rerun this cell."
    )

else:

    deleted_directories = []

    for directory_text in empty_case_df["path"]:

        case_directory = Path(
            directory_text
        ).resolve()

        # Repeat safety checks immediately before deletion
        if (
            case_directory.parent
            != simulation_root
        ):
            raise RuntimeError(
                f"Refusing to delete unsafe path: "
                f"{case_directory}"
            )

        if not case_directory.name.startswith("Re"):
            raise RuntimeError(
                f"Unexpected directory name: "
                f"{case_directory.name}"
            )

        if (
            case_directory.exists()
            and not case_directory.is_symlink()
        ):
            shutil.rmtree(case_directory)

            deleted_directories.append(
                case_directory.name
            )

            print(
                f"Deleted: "
                f"{case_directory.name}"
            )

    print()
    print(
        f"Deleted {len(deleted_directories)} "
        f"simulation directories."
    )